# Comprehensive Portfolio Viewer
## FIFO-based P&L Analysis with Interest Cost Calculation

This notebook analyzes your complete portfolio history including:
- Market trades (buy/sell)
- IPO allocations
- Bonus shares
- Rights shares
- Interest cost analysis (opportunity cost)
- Realized & Unrealized P&L

In [ ]:
import sys
from pathlib import Path

from src.core.portfolio.analyzer import PortfolioAnalyzer
from src.config.settings import config
import pandas as pd
from pathlib import Path

In [ ]:
# Configuration
username = '3522757'  # Change this to your username

# Initialize analyzer
analyzer = PortfolioAnalyzer(username)

## Portfolio Summary
Complete overview of all scripts with realized and unrealized P&L

In [ ]:
# Load current prices from Wacc Rates
wacc_path = Path(analyzer.base_dir, 'Wacc Rates.csv')
current_prices = {}
if wacc_path.exists():
    wacc_df = pd.read_csv(wacc_path)
    for _, row in wacc_df.iterrows():
        if pd.notna(row.get('LTP')):
            current_prices[row['Scrip']] = float(row['LTP'])

# Generate portfolio summary
portfolio_summary = analyzer.get_portfolio_summary(current_prices)
portfolio_summary

In [ ]:
# Portfolio Totals
print("="*70)
print("PORTFOLIO TOTALS")
print("="*70)
print(f"Total Investment:         Rs. {portfolio_summary['Total Investment'].sum():,.2f}")
print(f"Current Value:            Rs. {portfolio_summary['Current Value'].sum():,.2f}")
print(f"Realized P&L:             Rs. {portfolio_summary['Realized P&L'].sum():,.2f}")
print(f"Unrealized P&L:           Rs. {portfolio_summary['Unrealized P&L'].sum():,.2f}")
print(f"Total P&L:                Rs. {portfolio_summary['Total P&L'].sum():,.2f}")
print(f"Interest Cost:            Rs. {portfolio_summary['Interest Cost'].sum():,.2f}")
print(f"Net P&L (After Interest): Rs. {portfolio_summary['Net P&L (After Interest)'].sum():,.2f}")
print(f"\nTotal Return:             {portfolio_summary['Total P&L'].sum() / portfolio_summary['Total Investment'].sum() * 100:.2f}%")
print(f"Net Return (After Interest): {portfolio_summary['Net P&L (After Interest)'].sum() / portfolio_summary['Total Investment'].sum() * 100:.2f}%")
print("="*70)

## Current Holdings
Detailed view of your current holdings

In [ ]:
current_holdings = analyzer.get_current_holdings_summary(current_prices)
current_holdings

## Detailed Holdings by Lot
See each individual purchase lot with its P&L and interest cost

In [ ]:
detailed_lots = analyzer.get_detailed_lots(current_prices)
detailed_lots

## Script-Specific Analysis
Deep dive into a specific script

In [ ]:
# Choose a script to analyze
scrip_to_analyze = 'NABIL'  # Change this

# Transaction history for this scrip
trans_history = analyzer.get_transaction_history()
scrip_transactions = trans_history[trans_history['Scrip'] == scrip_to_analyze]
print(f"\nTransaction History for {scrip_to_analyze}:")
print(scrip_transactions.to_string(index=False))

# Current lots for this scrip
scrip_lots = detailed_lots[detailed_lots['Scrip'] == scrip_to_analyze]
print(f"\nCurrent Holdings Lots for {scrip_to_analyze}:")
print(scrip_lots.to_string(index=False))

# Summary for this scrip
scrip_summary = portfolio_summary[portfolio_summary['Scrip'] == scrip_to_analyze]
print(f"\nSummary for {scrip_to_analyze}:")
print(scrip_summary.to_string(index=False))

## Interest Cost Analysis
See which holdings have the highest opportunity cost

In [ ]:
interest_analysis = analyzer.get_interest_analysis()
interest_analysis

## Complete Transaction History
Chronological list of all transactions

In [ ]:
transaction_history = analyzer.get_transaction_history()
transaction_history

## Export Reports
Generate all CSV reports

In [ ]:
# Generate all reports
reports = analyzer.generate_reports()
print("\nReports generated successfully!")
print(f"Location: {analyzer.base_dir}")

## Best & Worst Performers
See your top and bottom performing stocks

In [ ]:
# Top 5 performers by Net Return %
print("Top 5 Performers (Net Return %):")
print(portfolio_summary.nlargest(5, 'Net Return %')[['Scrip', 'Total Investment', 'Net P&L (After Interest)', 'Net Return %']].to_string(index=False))

print("\n" + "="*70 + "\n")

# Bottom 5 performers by Net Return %
print("Bottom 5 Performers (Net Return %):")
print(portfolio_summary.nsmallest(5, 'Net Return %')[['Scrip', 'Total Investment', 'Net P&L (After Interest)', 'Net Return %']].to_string(index=False))

## Portfolio Statistics

In [ ]:
print("Portfolio Statistics:")
print(f"Total Scripts: {len(portfolio_summary)}")
print(f"Scripts with Current Holdings: {len(current_holdings)}")
print(f"Scripts Fully Sold: {len(portfolio_summary[portfolio_summary['Current Holdings'] == 0])}")
print(f"Total Transactions: {len(transaction_history)}")
print(f"\nProfit-making Scripts: {len(portfolio_summary[portfolio_summary['Total P&L'] > 0])}")
print(f"Loss-making Scripts: {len(portfolio_summary[portfolio_summary['Total P&L'] < 0])}")
print(f"Break-even Scripts: {len(portfolio_summary[portfolio_summary['Total P&L'] == 0])}")